<a href="https://colab.research.google.com/github/norewyx0205/vlm-event-boundary/blob/main/notebooks/colab_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ladder Event Boundary Evaluation on Colab

This notebook keeps the baseline sanity-check evaluation and runs the 6-level ladder experiment with Qwen3-VL.


In [ ]:
%cd /content
!ls

In [ ]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN")

## Clone or Update Repository

If the repository already exists in Colab, this cell pulls the latest code. If it does not exist, it clones the repo.


In [ ]:
from getpass import getpass
import os

REPO_URL = "github.com/norewyx0205/vlm-event-boundary.git"
REPO_DIR = "/content/vlm-event-boundary"

if not os.path.exists(REPO_DIR):
    token = os.environ["GH_TOKEN"]
    if token:
        !git clone https://{token}@{REPO_URL} {REPO_DIR}
    else:
        !git clone https://{REPO_URL} {REPO_DIR}
else:
    print("Repository already exists; pulling latest changes...")
    %cd {REPO_DIR}
    !git pull --ff-only

# check repo version
!git -C /content/vlm-event-boundary rev-parse --short HEAD
!grep -n "subprocess.Popen" /content/vlm-event-boundary/notebooks/colab_eval.ipynb


In [ ]:
%cd /content/vlm-event-boundary
!ls

## Install Dependencies

These packages are needed for Qwen video input, video generation, and result analysis.


In [ ]:
%pip install "transformers==5.9.0" accelerate "qwen-vl-utils==0.0.14" "decord==0.6.0" opencv-python imageio-ffmpeg

import subprocess
import sys
subprocess_transformers = subprocess.check_output(
    [sys.executable, "-c", "import transformers; print(transformers.__version__)"],
    text=True,
).strip()
if subprocess_transformers != "5.9.0":
    raise RuntimeError(
        f"Expected subprocess transformers 5.9.0, found {subprocess_transformers}."
    )
print("Pinned Transformers subprocess runtime:", subprocess_transformers)

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    torch.set_default_device("cuda")


## Configuration

Qwen3-VL is the default model. Select one pipeline profile and optionally override individual experiments. Completed experiments should normally use `reuse`; only `run` may load Qwen or regenerate videos.


In [ ]:
import json
from pathlib import Path

from scripts.common import slugify
from scripts.experiment_artifacts import (
    announce_experiment,
    discover_latest_archive,
    latest_match,
    latest_results_by_dataset,
    mode_analyzes,
    mode_runs,
    require_matches,
    require_paths,
    resolve_experiment_modes,
    restore_artifact_archive,
    stable_fingerprint,
)

PROJECT_ROOT = Path("/content/vlm-event-boundary")
MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"
MODEL_REVISION = "0c351dd01ed87e9c1b53cbc748cba10e6187ff3b"
EVAL_SEED = 42
ATTN_IMPLEMENTATION = "eager"
RESULT_DIR = str(PROJECT_ROOT / "results")
MODEL_RESULT_ROOT = Path(RESULT_DIR) / slugify(MODEL_NAME)

BASELINE_ANNOTATION = str(PROJECT_ROOT / "baseline_boundary_videos/annotations.jsonl")
SYNTHETIC_ANNOTATION = str(PROJECT_ROOT / "synthetic_boundary_videos/annotations.jsonl")
LADDER_ROOT = str(PROJECT_ROOT / "data/ladder_v2")
DATASET_VERSION = "ladder_v2"

# Profiles: part1_reuse, analysis_only, full_reproduction, smoke.
PIPELINE_PROFILE = "part1_reuse"
EXPERIMENT_MODE_OVERRIDES = {
    # Each value may be skip, reuse, analyze, or run when supported.
    # "baseline": "skip",
    # "synthetic": "skip",
    # "ladder": "reuse",
    # "ladder_smoke": "run",
    # "feature_ablation": "reuse",
    # "size_stress": "reuse",
    # "size_clear_contrast": "reuse",
    # "diagnostics": "reuse",
    # "roi_perturbation": "analyze",
    # "attention_phase0": "reuse",
    # "attention_phase1": "analyze",
}
EXPERIMENT_MODES = resolve_experiment_modes(
    PIPELINE_PROFILE, EXPERIMENT_MODE_OVERRIDES
)

# Add exact uploaded ZIP paths here. With an empty list, the latest matching
# real result archive in /content is restored automatically when available.
ARTIFACT_ARCHIVES = []
AUTO_DISCOVER_LATEST_ARTIFACT = True
ARTIFACT_SEARCH_ROOTS = [Path("/content")]
RESTORED_ARTIFACTS = []
archive_paths = [Path(path) for path in ARTIFACT_ARCHIVES]
if not archive_paths and AUTO_DISCOVER_LATEST_ARTIFACT:
    discovered = discover_latest_archive(ARTIFACT_SEARCH_ROOTS)
    if discovered is not None:
        archive_paths = [discovered]
for archive_path in archive_paths:
    restored = restore_artifact_archive(
        archive_path,
        PROJECT_ROOT,
        expected={
            "model_name": MODEL_NAME,
            "model_revision": MODEL_REVISION,
        },
    )
    RESTORED_ARTIFACTS.append(restored)
    print(f"Restored real artifacts from {archive_path.name}")
if not RESTORED_ARTIFACTS:
    print("No archive restored. Reuse stages will validate repository/local artifacts before use.")
print("Pipeline profile:", PIPELINE_PROFILE)
print(json.dumps(EXPERIMENT_MODES, indent=2))

## Generate Baseline and Synthetic Reference Datasets

This stage validates the frozen baseline/synthetic datasets in `reuse` mode and regenerates them only when either experiment is explicitly set to `run`.

In [ ]:
reference_modes = {
    "baseline": EXPERIMENT_MODES["baseline"],
    "synthetic": EXPERIMENT_MODES["synthetic"],
}
for name, mode in reference_modes.items():
    announce_experiment(name, mode)
if any(mode_runs(mode) for mode in reference_modes.values()):
    subprocess.run([
        "python", "generate_2d_boundary_videos.py", "--dataset", "all"
    ], check=True)
for name, annotation in [
    ("baseline", BASELINE_ANNOTATION),
    ("synthetic", SYNTHETIC_ANNOTATION),
]:
    if reference_modes[name] != "skip":
        require_paths(
            name, [annotation],
            hint="Upload a matching real artifact archive or set this experiment to run.",
        )

## Baseline Sanity Check

This keeps the earlier simple baseline. It should verify that Qwen3 can solve the easy before/after task.


In [ ]:
from pathlib import Path

for name, annotation in [
    ("baseline", BASELINE_ANNOTATION),
    ("synthetic", SYNTHETIC_ANNOTATION),
]:
    if EXPERIMENT_MODES[name] == "skip":
        continue
    path = Path(annotation)
    print(
        f"{name}: eval_rows={sum(1 for _ in open(path))}; "
        f"videos={len(list((path.parent / 'videos').glob('*.mp4')))}"
    )

In [ ]:
mode = EXPERIMENT_MODES["baseline"]
if mode_runs(mode):
    subprocess.run([
        "python", "scripts/run_eval.py",
        "--annotation_path", BASELINE_ANNOTATION,
        "--model_name", MODEL_NAME,
        "--model_revision", MODEL_REVISION,
        "--seed", str(EVAL_SEED), "--deterministic",
        "--attn_implementation", ATTN_IMPLEMENTATION,
        "--dataset_name", "baseline_qwen3_sanity_check",
        "--output_dir", RESULT_DIR,
    ], check=True)
if mode != "skip":
    BASELINE_RESULT = latest_match(
        "baseline", MODEL_RESULT_ROOT,
        "baseline_qwen3_sanity_check/*/raw_results.jsonl",
        hint="Restore an archived baseline run or set baseline=run.",
    )
    print("Baseline result:", BASELINE_RESULT)
else:
    print("Baseline evaluation skipped.")

## Synthetic Hard Reference Evaluation

This evaluates the legacy harder synthetic set so it can be compared with the simple baseline and the ladder levels.

In [ ]:
mode = EXPERIMENT_MODES["synthetic"]
if mode_runs(mode):
    subprocess.run([
        "python", "scripts/run_eval.py",
        "--annotation_path", SYNTHETIC_ANNOTATION,
        "--model_name", MODEL_NAME,
        "--model_revision", MODEL_REVISION,
        "--seed", str(EVAL_SEED), "--deterministic",
        "--attn_implementation", ATTN_IMPLEMENTATION,
        "--dataset_name", "synthetic_qwen3_reference",
        "--output_dir", RESULT_DIR,
    ], check=True)
if mode != "skip":
    SYNTHETIC_RESULT = latest_match(
        "synthetic", MODEL_RESULT_ROOT,
        "synthetic_qwen3_reference/*/raw_results.jsonl",
        hint="Restore an archived synthetic run or set synthetic=run.",
    )
    print("Synthetic result:", SYNTHETIC_RESULT)
else:
    print("Synthetic evaluation skipped.")

## Generate 6-Level Ladder Dataset

This validates the 6-level dataset in `reuse`/`analyze` mode and regenerates it only when `ladder=run`.


In [ ]:
mode = EXPERIMENT_MODES["ladder"]
announce_experiment("ladder", mode)
if mode_runs(mode):
    subprocess.run([
        "python", "scripts/generate_ladder_dataset.py",
        "--dataset_version", DATASET_VERSION,
        "--samples_per_level", "30",
        "--output_root", LADDER_ROOT,
        "--seed", "42",
    ], check=True)
if mode != "skip":
    LADDER_ANNOTATIONS = require_matches(
        "ladder", LADDER_ROOT, "level_*/annotations.jsonl", minimum=6,
        hint="Restore the ladder dataset or set ladder=run.",
    )
else:
    LADDER_ANNOTATIONS = []

## Check Ladder Dataset

Each level should contain 30 base samples × 4 boundary conditions × 2 mirrored prompts = 240 evaluation rows. The full ladder has 6 levels.


In [ ]:
from pathlib import Path

if EXPERIMENT_MODES["ladder"] != "skip":
    for ann in LADDER_ANNOTATIONS:
        video_count = len(list((ann.parent / "videos").glob("*.mp4")))
        row_count = sum(1 for _ in open(ann))
        print(ann.parent.name, "videos=", video_count, "eval_rows=", row_count)
else:
    print("Ladder dataset check skipped.")


## Qwen3 Ladder Smoke Test

Run a tiny subset before launching the full ladder evaluation.


In [ ]:
smoke_mode = EXPERIMENT_MODES["ladder_smoke"]
announce_experiment("ladder_smoke", smoke_mode)
if mode_runs(smoke_mode):
    smoke_annotation = Path(LADDER_ROOT) / "level_1_simple/annotations.jsonl"
    require_paths("ladder_smoke", [smoke_annotation])
    subprocess.run([
        "python", "scripts/run_eval.py",
        "--annotation_path", str(smoke_annotation),
        "--model_name", MODEL_NAME,
        "--model_revision", MODEL_REVISION,
        "--seed", str(EVAL_SEED), "--deterministic",
        "--attn_implementation", ATTN_IMPLEMENTATION,
        "--dataset_name", "smoke_ladder_v2_level_1_simple_qwen3",
        "--output_dir", RESULT_DIR, "--max_samples", "4",
    ], check=True)

## Run Qwen3 on All 6 Ladder Levels

This is the main ladder experiment. Results are saved under `results/<safe_model_name>/<dataset_name>/<timestamp>/`.


In [ ]:
mode = EXPERIMENT_MODES["ladder"]
if mode_runs(mode):
    subprocess.run([
        "python", "scripts/run_eval.py",
        "--annotation_root", LADDER_ROOT,
        "--model_name", MODEL_NAME,
        "--model_revision", MODEL_REVISION,
        "--seed", str(EVAL_SEED), "--deterministic",
        "--attn_implementation", ATTN_IMPLEMENTATION,
        "--dataset_name_prefix", f"{DATASET_VERSION}_",
        "--output_dir", RESULT_DIR,
    ], check=True)
if mode != "skip":
    LADDER_RAW_RESULTS = latest_results_by_dataset(
        "ladder", MODEL_RESULT_ROOT,
        f"{DATASET_VERSION}_level_*", minimum=6,
        hint="Restore a completed six-level run or set ladder=run.",
    )
    print(f"Ladder raw results ready: {len(LADDER_RAW_RESULTS)} files")

## Analyze Ladder Results

This aggregates all Qwen3 ladder runs and treats prompt Accuracy and Strict both-correct pair accuracy as co-primary metrics. It produces both 6-level curves, direct Accuracy-vs-Strict comparisons, paired boundary comparisons, and swap-consistency diagnostics.



In [ ]:
ANALYSIS_DIR = str(PROJECT_ROOT / f"analysis/{DATASET_VERSION}_ladder")
mode = EXPERIMENT_MODES["ladder"]
if mode_analyzes(mode):
    subprocess.run([
        "python", "scripts/analyze_results.py",
        "--input", RESULT_DIR,
        "--dataset_name_prefix", f"{DATASET_VERSION}_level_",
        "--latest_per_dataset",
        "--output_dir", ANALYSIS_DIR, "--plots",
    ], check=True)
elif mode == "reuse":
    require_paths("ladder", [Path(ANALYSIS_DIR) / "summary.json"])
    print("Reusing ladder analysis:", ANALYSIS_DIR)
else:
    print("Ladder analysis skipped.")

## Inspect Saved Files


In [ ]:
artifact_status = {
    "pipeline_profile": PIPELINE_PROFILE,
    "restored_archives": [
        Path(item["archive_path"]).name for item in RESTORED_ARTIFACTS
    ],
    "raw_result_files": len(list(Path(RESULT_DIR).glob("*/*/*/raw_results.jsonl"))),
    "analysis_files": len(list((PROJECT_ROOT / "analysis").glob("**/*"))),
}
print(json.dumps(artifact_status, indent=2))


## Level 5 Feature-Ablation Pilot

This pilot compares four structurally paired Level 5 variants: full, shape-only, color-only, and size-only. It also includes a separate size-only 2x2 stress pilot crossing absolute target size with distractor count.


In [ ]:
ABLATION_VERSION = "l5_feature_ablation_v1"
ABLATION_ROOT = f"/content/vlm-event-boundary/data/{ABLATION_VERSION}"
ABLATION_ANALYSIS_DIR = f"/content/vlm-event-boundary/analysis/{ABLATION_VERSION}"
SIZE_STRESS_ROOT = f"{ABLATION_ROOT}/size_stress_pilot"
SIZE_STRESS_ANALYSIS_DIR = f"{ABLATION_ANALYSIS_DIR}_size_stress"
SIZE_CLEAR_CONTRAST_ROOT = f"{ABLATION_ROOT}/size_clear_contrast_pilot"
SIZE_CLEAR_CONTRAST_ANALYSIS_DIR = f"{ABLATION_ANALYSIS_DIR}_size_clear_contrast"
DIAGNOSTIC_ROOT = "/content/vlm-event-boundary/data/diagnostics"
SIZE_CLEAR_DIAGNOSTIC_ANNOTATION = f"{DIAGNOSTIC_ROOT}/l5_size_clear_contrast_diagnostics/annotations.jsonl"
SIZE_CLEAR_DIAGNOSTIC_ANALYSIS_DIR = f"{ABLATION_ANALYSIS_DIR}_size_clear_contrast_diagnostics"
PERTURBATION_ROOT = "/content/vlm-event-boundary/data/perturbations/l5_clear_small_many"
PERTURBATION_ANALYSIS_DIR = f"{ABLATION_ANALYSIS_DIR}_perturb_l5_clear_small_many"
ATTENTION_OUTPUT_PATH = "/content/vlm-event-boundary/analysis/attention/l5_clear_small_many_attention.json"
ATTENTION_VISUALIZATION_DIR = "/content/vlm-event-boundary/analysis/attention/l5_clear_small_many_figures"
ATTENTION_ANALYSIS_DIR = "/content/vlm-event-boundary/analysis/attention/l5_clear_small_many_metrics"
FEATURE_ATTENTION_OUTPUT_PATH = "/content/vlm-event-boundary/analysis/attention/l5_feature_calibration_attention.json"
FEATURE_ATTENTION_CASE_MANIFEST = "/content/vlm-event-boundary/analysis/attention/l5_feature_calibration_cases.jsonl"
FEATURE_ATTENTION_ANALYSIS_DIR = "/content/vlm-event-boundary/analysis/attention/l5_feature_calibration_metrics"
FEATURE_ATTENTION_BASE_SAMPLES = 4
FEATURE_ATTENTION_MAX_SAMPLES = FEATURE_ATTENTION_BASE_SAMPLES * 4 * 4 * 2
ABLATION_VARIANTS = ["L5_full", "L5_shape_only", "L5_color_only", "L5_size_only"]


### Generate And Validate Paired Stimuli


In [ ]:
feature_mode = EXPERIMENT_MODES["feature_ablation"]
stress_mode = EXPERIMENT_MODES["size_stress"]
announce_experiment("feature_ablation", feature_mode)
announce_experiment("size_stress", stress_mode)
if mode_runs(feature_mode) or mode_runs(stress_mode):
    subprocess.run([
        "python", "scripts/generate_l5_feature_ablation.py",
        "--dataset_version", ABLATION_VERSION,
        "--samples_per_variant", "30",
        "--size_stress_samples_per_cell", "10",
        "--output_root", ABLATION_ROOT, "--seed", "42",
    ], check=True)
    subprocess.run([
        "python", "scripts/check_l5_feature_ablation.py",
        "--root", ABLATION_ROOT,
    ], check=True)
if feature_mode != "skip":
    FEATURE_ANNOTATIONS = require_matches(
        "feature_ablation", ABLATION_ROOT, "L5_*/annotations.jsonl", minimum=4,
        hint="Restore the feature dataset or set feature_ablation=run.",
    )
if stress_mode != "skip":
    SIZE_STRESS_ANNOTATIONS = require_matches(
        "size_stress", SIZE_STRESS_ROOT, "*/annotations.jsonl", minimum=4,
        hint="Restore the size-stress dataset or set size_stress=run.",
    )

### Run Qwen3 Evaluation

Each variant retains all four boundary conditions and original/swapped mirrored prompts.


In [ ]:
mode = EXPERIMENT_MODES["feature_ablation"]
if mode_runs(mode):
    subprocess.run([
        "python", "scripts/run_eval.py",
        "--annotation_root", ABLATION_ROOT,
        "--model_name", MODEL_NAME, "--model_revision", MODEL_REVISION,
        "--seed", str(EVAL_SEED), "--deterministic",
        "--attn_implementation", ATTN_IMPLEMENTATION,
        "--dataset_name_prefix", f"{ABLATION_VERSION}_main_",
        "--output_dir", RESULT_DIR,
    ], check=True)
if mode != "skip":
    FEATURE_RAW_RESULTS = latest_results_by_dataset(
        "feature_ablation", MODEL_RESULT_ROOT,
        f"{ABLATION_VERSION}_main_*", minimum=4,
        hint="Restore the completed feature run or set feature_ablation=run.",
    )
    print(f"Feature raw results ready: {len(FEATURE_RAW_RESULTS)} files")

### Run Size-Only 2x2 Stress Pilot

This evaluates 10 base samples in each of large/few, large/many, small/few, and small/many, for 320 prompt evaluations in total.


In [ ]:
mode = EXPERIMENT_MODES["size_stress"]
if mode_runs(mode):
    subprocess.run([
        "python", "scripts/run_eval.py",
        "--annotation_root", SIZE_STRESS_ROOT,
        "--model_name", MODEL_NAME, "--model_revision", MODEL_REVISION,
        "--seed", str(EVAL_SEED), "--deterministic",
        "--attn_implementation", ATTN_IMPLEMENTATION,
        "--dataset_name_prefix", f"{ABLATION_VERSION}_size_stress_",
        "--output_dir", RESULT_DIR,
    ], check=True)
if mode != "skip":
    SIZE_STRESS_RAW_RESULTS = latest_results_by_dataset(
        "size_stress", MODEL_RESULT_ROOT,
        f"{ABLATION_VERSION}_size_stress_*", minimum=4,
        hint="Restore the size-stress run or set size_stress=run.",
    )
    print(f"Size-stress raw results ready: {len(SIZE_STRESS_RAW_RESULTS)} files")

### Analyze Feature, Boundary, And Position Effects

The analysis uses the latest run for each variant and writes prompt accuracy, strict mirrored-pair accuracy, the accuracy-strict gap `d`, position-sensitive pair rates, paired comparisons, swap consistency, and report-ready plots for the main research questions.


In [ ]:
mode = EXPERIMENT_MODES["feature_ablation"]
if mode_analyzes(mode):
    subprocess.run([
        "python", "scripts/analyze_results.py",
        "--input", RESULT_DIR,
        "--dataset_name_prefix", f"{ABLATION_VERSION}_main_",
        "--latest_per_dataset", "--output_dir", ABLATION_ANALYSIS_DIR, "--plots",
    ], check=True)
elif mode == "reuse":
    require_paths("feature_ablation", [Path(ABLATION_ANALYSIS_DIR) / "summary.json"])
    print("Reusing feature analysis:", ABLATION_ANALYSIS_DIR)
else:
    print("Feature analysis skipped.")

### Analyze Size And Crowding Effects

This produces cell-level accuracy, strict mirrored-pair accuracy, boundary-condition plots, and the large-vs-small, few-vs-many, and interaction estimates.


In [ ]:
mode = EXPERIMENT_MODES["size_stress"]
if mode_analyzes(mode):
    subprocess.run([
        "python", "scripts/analyze_results.py",
        "--input", RESULT_DIR,
        "--dataset_name_prefix", f"{ABLATION_VERSION}_size_stress_",
        "--latest_per_dataset", "--output_dir", SIZE_STRESS_ANALYSIS_DIR, "--plots",
    ], check=True)
elif mode == "reuse":
    require_paths("size_stress", [Path(SIZE_STRESS_ANALYSIS_DIR) / "summary.json"])
    print("Reusing size-stress analysis:", SIZE_STRESS_ANALYSIS_DIR)
else:
    print("Size-stress analysis skipped.")

### Generate Size-Only Clear-Contrast Pilot

This is the final Part 1 size-only contrast check. It repeats the 2x2 size/crowding design with clearer target-distractor size margins so we can test whether the previous size-only pattern survives when the smallest/largest distinction is visually obvious.


In [ ]:
mode = EXPERIMENT_MODES["size_clear_contrast"]
announce_experiment("size_clear_contrast", mode)
if mode_runs(mode):
    subprocess.run([
        "python", "scripts/generate_l5_feature_ablation.py",
        "--dataset_version", ABLATION_VERSION,
        "--size_clear_contrast_only",
        "--size_clear_contrast_samples_per_cell", "10",
        "--output_root", ABLATION_ROOT, "--seed", "42",
    ], check=True)
    subprocess.run([
        "python", "scripts/check_l5_feature_ablation.py", "--root", ABLATION_ROOT,
    ], check=True)
if mode != "skip":
    SIZE_CLEAR_ANNOTATIONS = require_matches(
        "size_clear_contrast", SIZE_CLEAR_CONTRAST_ROOT,
        "*/annotations.jsonl", minimum=4,
        hint="Restore the clear-contrast dataset or set size_clear_contrast=run.",
    )

### Run Size-Only Clear-Contrast Pilot

This evaluates four clear-contrast scenes: clear_large_few, clear_large_many, clear_small_few, and clear_small_many. Each scene has 10 base samples, four boundary conditions, and original/swapped mirrored prompts.


In [ ]:
mode = EXPERIMENT_MODES["size_clear_contrast"]
if mode_runs(mode):
    subprocess.run([
        "python", "scripts/run_eval.py",
        "--annotation_root", SIZE_CLEAR_CONTRAST_ROOT,
        "--model_name", MODEL_NAME, "--model_revision", MODEL_REVISION,
        "--seed", str(EVAL_SEED), "--deterministic",
        "--attn_implementation", ATTN_IMPLEMENTATION,
        "--dataset_name_prefix", f"{ABLATION_VERSION}_size_clear_contrast_",
        "--output_dir", RESULT_DIR,
    ], check=True)
if mode != "skip":
    SIZE_CLEAR_RAW_RESULTS = latest_results_by_dataset(
        "size_clear_contrast", MODEL_RESULT_ROOT,
        f"{ABLATION_VERSION}_size_clear_contrast_*", minimum=4,
        hint="Restore the clear-contrast run or set size_clear_contrast=run.",
    )
    print(f"Clear-contrast raw results ready: {len(SIZE_CLEAR_RAW_RESULTS)} files")

### Analyze Clear-Contrast Size Effects

This produces the same accuracy, strict both-correct, boundary, size/crowding, and interaction summaries as the original size stress pilot, but for the clearer size contrast stimuli.


In [ ]:
mode = EXPERIMENT_MODES["size_clear_contrast"]
if mode_analyzes(mode):
    subprocess.run([
        "python", "scripts/analyze_results.py",
        "--input", RESULT_DIR,
        "--dataset_name_prefix", f"{ABLATION_VERSION}_size_clear_contrast_",
        "--latest_per_dataset",
        "--output_dir", SIZE_CLEAR_CONTRAST_ANALYSIS_DIR, "--plots",
    ], check=True)
elif mode == "reuse":
    require_paths(
        "size_clear_contrast",
        [Path(SIZE_CLEAR_CONTRAST_ANALYSIS_DIR) / "summary.json"],
    )
    print("Reusing clear-contrast analysis:", SIZE_CLEAR_CONTRAST_ANALYSIS_DIR)
else:
    print("Clear-contrast analysis skipped.")

## Mechanism Probes: Diagnostics, Perturbation, And Attention

These cells are optional and should be run after the clear-contrast evaluation. They implement the two-layer method: first behavioral diagnostics and causal perturbations, then small-sample attention/ROI probing. They are disabled by default so a normal Run All does not accidentally add a long diagnostic run.


### Create Diagnostic Prompts

Diagnostic prompts reuse the same videos but ask simpler questions about object identity, motion binding, and first-mover identity. This helps separate `recognise objects` from `track objects` and final before/after reasoning.


In [ ]:
import subprocess

mode = EXPERIMENT_MODES["diagnostics"]
announce_experiment("diagnostics", mode)
if mode_runs(mode):
    subprocess.run([
        "python", "scripts/make_diagnostic_annotations.py",
        "--annotation_root", SIZE_CLEAR_CONTRAST_ROOT,
        "--output_path", SIZE_CLEAR_DIAGNOSTIC_ANNOTATION,
    ], check=True)
    subprocess.run([
        "python", "scripts/run_eval.py",
        "--annotation_path", SIZE_CLEAR_DIAGNOSTIC_ANNOTATION,
        "--model_name", MODEL_NAME,
        "--model_revision", MODEL_REVISION,
        "--seed", str(EVAL_SEED),
        "--deterministic",
        "--attn_implementation", ATTN_IMPLEMENTATION,
        "--dataset_name", "l5_size_clear_contrast_diagnostics",
        "--output_dir", RESULT_DIR,
    ], check=True)
if mode != "skip":
    require_paths("diagnostics", [SIZE_CLEAR_DIAGNOSTIC_ANNOTATION])
    DIAGNOSTIC_RAW_RESULT = latest_match(
        "diagnostics", MODEL_RESULT_ROOT,
        "l5_size_clear_contrast_diagnostics/*/raw_results.jsonl",
        hint="Restore the diagnostic run or set diagnostics=run.",
    )
if mode_analyzes(mode):
    subprocess.run([
        "python", "scripts/analyze_results.py",
        "--input", RESULT_DIR,
        "--dataset_name_prefix", "l5_size_clear_contrast_diagnostics",
        "--latest_per_dataset",
        "--output_dir", SIZE_CLEAR_DIAGNOSTIC_ANALYSIS_DIR,
        "--plots",
    ], check=True)
    subprocess.run([
        "find", SIZE_CLEAR_DIAGNOSTIC_ANALYSIS_DIR,
        "-maxdepth", "1", "-type", "f",
    ], check=True)
elif mode == "reuse":
    require_paths(
        "diagnostics",
        [Path(SIZE_CLEAR_DIAGNOSTIC_ANALYSIS_DIR) / "summary.json"],
    )
    print("Reusing diagnostic analysis:", SIZE_CLEAR_DIAGNOSTIC_ANALYSIS_DIR)
else:
    print("Diagnostic experiment skipped.")


### Causal ROI Perturbation Probe

In `run` mode this creates codec-matched controls, ROI masks, fixed-duration temporal-gap interventions, QA previews, model results, and analysis for a small subset of `clear_small_many`. In `reuse` mode it only validates the archived artifacts. Matched comparisons use `reencode_control` as the primary baseline, while `original` versus `reencode_control` measures codec effects.


In [ ]:
import subprocess
from pathlib import Path
from IPython.display import Image, display

mode = EXPERIMENT_MODES["roi_perturbation"]
announce_experiment("roi_perturbation", mode)
PERTURBATION_SOURCE = f"{SIZE_CLEAR_CONTRAST_ROOT}/L5_size_only_clear_small_many/annotations.jsonl"
PERTURBATION_MAX_BASE_SAMPLES = 4  # technical pilot; increase only after control checks pass
PERTURBATION_TYPES = (
    "original,reencode_control,mask_target_1,mask_target_2,mask_distractors,"
    "mask_background_control,remove_visual_marker,gap_removed,gap_shortened,gap_shifted"
)
ROI_MASK_PADDING = 6
ROI_MASK_SCOPE = "all_frames"  # use motion_window to isolate motion evidence

def run_streaming(command, label):
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"{label} failed with exit code {return_code}; see the log above.")

if mode_runs(mode):
    run_streaming([
        "python", "scripts/make_roi_perturbation_dataset.py",
        "--annotation_path", PERTURBATION_SOURCE,
        "--output_root", PERTURBATION_ROOT,
        "--max_base_samples", str(PERTURBATION_MAX_BASE_SAMPLES),
        "--perturbations", PERTURBATION_TYPES,
        "--mask_padding", str(ROI_MASK_PADDING),
        "--mask_mode", "dynamic",
        "--mask_scope", ROI_MASK_SCOPE,
        "--sham_reference", "distractors",
        "--sham_clearance", "4",
        "--sham_max_path_relative_error", "0.10",
    ], "ROI perturbation build")
    for preview_condition in ["visual_boundary", "temporal_boundary"]:
        preview_path = f"{PERTURBATION_ANALYSIS_DIR}/roi_qa_{preview_condition}.png"
        subprocess.run([
            "python", "scripts/visualize_roi_perturbations.py",
            "--annotation_path", f"{PERTURBATION_ROOT}/annotations.jsonl",
            "--condition", preview_condition,
            "--output_path", preview_path,
        ], check=True)
        display(Image(filename=preview_path))

    perturbation_annotation = Path(PERTURBATION_ROOT) / "annotations.jsonl"
    require_paths("roi_perturbation", [perturbation_annotation])
    subprocess.run([
        "python", "scripts/run_eval.py",
        "--annotation_path", str(perturbation_annotation),
        "--model_name", MODEL_NAME,
        "--model_revision", MODEL_REVISION,
        "--seed", str(EVAL_SEED),
        "--deterministic",
        "--attn_implementation", ATTN_IMPLEMENTATION,
        "--dataset_name", "l5_clear_small_many_perturbation",
        "--output_dir", RESULT_DIR,
    ], check=True)
if mode != "skip":
    perturbation_annotation = Path(PERTURBATION_ROOT) / "annotations.jsonl"
    require_paths(
        "roi_perturbation",
        [
            perturbation_annotation,
            Path(PERTURBATION_ROOT) / "manifest.json",
            Path(PERTURBATION_ROOT) / "perturbation_stats.jsonl",
        ],
        hint="Restore the perturbation artifacts or set roi_perturbation=run.",
    )
    PERTURBATION_RAW_RESULT = latest_match(
        "roi_perturbation", MODEL_RESULT_ROOT,
        "l5_clear_small_many_perturbation/*/raw_results.jsonl",
        hint="Restore the perturbation evaluation or set roi_perturbation=run.",
    )
if mode_analyzes(mode):
    subprocess.run([
        "python", "scripts/analyze_results.py",
        "--input", RESULT_DIR,
        "--dataset_name_prefix", "l5_clear_small_many_perturbation",
        "--latest_per_dataset",
        "--output_dir", PERTURBATION_ANALYSIS_DIR,
        "--plots",
    ], check=True)
elif mode == "reuse":
    require_paths(
        "roi_perturbation",
        [Path(PERTURBATION_ANALYSIS_DIR) / "summary.json"],
    )
    print("Reusing ROI perturbation analysis:", PERTURBATION_ANALYSIS_DIR)
else:
    print("ROI perturbation experiment skipped.")


### Attention ROI Probe

This probes the final prompt position whose logits predict the first A/B answer token. A prefix cache keeps the query to one token, and the script requires parity with standard greedy generation. Cases are selected from archived behavioural outcomes rather than annotation order; temporal-versus-visual contrasts are retained as atomic matched bundles. ROI attribution uses fractional cell overlap, averages every sampled frame within each temporal patch, preserves mixed phase weights, and records padding sensitivity.

The plots now separate visual-normalised attention mass from area-normalised enrichment. Target labels include semantic identity, mover order, and grammatical subject role. After the GPU probe, a CPU-only Phase 0 analysis writes layer-level and early/middle/late target contrast tables. Enrichment is not total attention, and attention remains a qualitative mechanism probe to interpret alongside perturbation results.


In [ ]:
import json
import subprocess
from pathlib import Path
from IPython.display import Image, display

mode = EXPERIMENT_MODES["attention_phase0"]
announce_experiment("attention_phase0", mode)
ATTENTION_ANNOTATIONS = f"{SIZE_CLEAR_CONTRAST_ROOT}/L5_size_only_clear_small_many/annotations.jsonl"
ATTENTION_CASE_MANIFEST = str(
    Path(ATTENTION_OUTPUT_PATH).with_name("l5_clear_small_many_cases.jsonl")
)
ATTENTION_MAX_SAMPLES = 8

if mode_runs(mode):
    main_runs = sorted(Path(RESULT_DIR).glob(
        "*/l5_feature_ablation_v1_size_clear_contrast_L5_size_only_clear_small_many/*/raw_results.jsonl"
    ))
    perturbation_runs = sorted(Path(RESULT_DIR).glob(
        "*/l5_clear_small_many_perturbation/*/raw_results.jsonl"
    ))
    if not main_runs or not perturbation_runs:
        raise FileNotFoundError(
            "Run the clear-small-many main evaluation and perturbation evaluation first."
        )
    subprocess.run([
        "python", "scripts/select_attention_cases.py",
        "--annotation_path", ATTENTION_ANNOTATIONS,
        "--main_results", str(main_runs[-1]),
        "--perturbation_results", str(perturbation_runs[-1]),
        "--output_path", ATTENTION_CASE_MANIFEST,
        "--max_video_pairs", str(ATTENTION_MAX_SAMPLES // 2),
    ], check=True)
    attention_command = [
        "python", "scripts/probe_attention_roi.py",
        "--annotation_path", ATTENTION_CASE_MANIFEST,
        "--output_path", ATTENTION_OUTPUT_PATH,
        "--visualization_dir", ATTENTION_VISUALIZATION_DIR,
        "--model_name", MODEL_NAME,
        "--expected_transformers_version", "5.9.0",
        "--model_revision", MODEL_REVISION,
        "--seed", str(EVAL_SEED),
        "--deterministic",
        "--attn_implementation", "eager",
        "--max_samples", str(ATTENTION_MAX_SAMPLES),
        "--roi_padding", "8",
        "--roi_assignment", "overlap",
        "--roi_padding_sensitivity", "0,4,8,12",
        "--parity_atol", "0.25",
        "--visualization_layer", "-1",
        "--head_reduction", "mean",
        "--empty_cache_each_sample",
    ]
    process = subprocess.Popen(
        attention_command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(
            f"Attention probe failed with exit code {return_code}; see the full log above."
        )
    rows = json.loads(Path(ATTENTION_OUTPUT_PATH).read_text())
    for row in rows[:3]:
        print(
            row.get("eval_id"),
            "prediction=", row.get("prediction"),
            "archived_match=", row.get("prediction_match"),
            "standard_token_match=", (row.get("decision_query") or {}).get("standard_first_token_match"),
            "max_logit_diff=", (row.get("decision_query") or {}).get("standard_logits_max_abs_diff"),
            "top10_overlap=", (row.get("decision_query") or {}).get("standard_top10_token_overlap"),
            "video_attention=", row.get("selected_layer_visual_attention_fraction"),
            "spatial_roi=", row.get("spatial_roi_attention"),
        )
    for figure_path in sorted(Path(ATTENTION_VISUALIZATION_DIR).glob("*.png"))[:6]:
        display(Image(filename=str(figure_path)))
elif mode in {"reuse", "analyze"}:
    require_paths(
        "attention_phase0",
        [
            ATTENTION_CASE_MANIFEST,
            ATTENTION_OUTPUT_PATH,
            Path(ATTENTION_OUTPUT_PATH).with_name(
                f"{Path(ATTENTION_OUTPUT_PATH).stem}_summary.json"
            ),
        ],
        hint="Restore the Phase 0 attention artifacts or set attention_phase0=run.",
    )
    print("Phase 0 attention JSON ready:", ATTENTION_OUTPUT_PATH)
else:
    print("Phase 0 attention probe skipped.")


### Phase 0 Attention Metric Decomposition

This CPU-only step can be rerun from an existing attention JSON without loading Qwen or using an A100. It separates total/all-token attention share, visual-normalised ROI mass, effective token-area share, mean attention per effective token, and area-normalised enrichment. It also writes Target 1 versus Target 2 layer and fixed early/middle/late contrast tables. This partition was specified before expanding to Phase 1 stimuli and remains fixed for subsequent analyses.


In [ ]:
import subprocess
from pathlib import Path
from IPython.display import Image, display

mode = EXPERIMENT_MODES["attention_phase0"]
if mode_analyzes(mode):
    require_paths("attention_phase0", [ATTENTION_OUTPUT_PATH])
    subprocess.run([
        "python", "scripts/analyze_attention_roi.py",
        "--input_path", ATTENTION_OUTPUT_PATH,
        "--output_dir", ATTENTION_ANALYSIS_DIR,
    ], check=True)
    subprocess.run([
        "python", "scripts/visualize_attention_roi.py",
        "--input_path", ATTENTION_OUTPUT_PATH,
        "--output_dir", ATTENTION_VISUALIZATION_DIR,
        "--project_root", "/content/vlm-event-boundary",
    ], check=True)
    print("Phase 0 tables:", sorted(path.name for path in Path(ATTENTION_ANALYSIS_DIR).glob("*")))
    contrast_figures = sorted(Path(ATTENTION_VISUALIZATION_DIR).glob("*_layer_target_contrasts.png"))
    for figure_path in contrast_figures[:4]:
        display(Image(filename=str(figure_path)))
elif mode == "reuse":
    require_paths(
        "attention_phase0",
        [
            Path(ATTENTION_ANALYSIS_DIR) / "summary.json",
            Path(ATTENTION_ANALYSIS_DIR) / "stage_target_contrasts.csv",
        ],
    )
    print("Reusing Phase 0 attention analysis:", ATTENTION_ANALYSIS_DIR)
else:
    print("Phase 0 attention analysis skipped.")


### Phase 1 Matched Feature Attention Calibration

This is the first expanded attention run. It selects four shared base stimuli across full, color-only, shape-only, and size-only; retains all four boundary conditions and both mirrored prompts; balances which target moves first; and checks that motion trajectories and event timing are structurally identical across feature variants. The default run contains 128 attention rows. Per-row plots are disabled to control storage; three aggregate mass-versus-enrichment figures are produced after the probe.


In [ ]:
import json
import subprocess
from pathlib import Path
from IPython.display import Image, display
from scripts.common import slugify

mode = EXPERIMENT_MODES["attention_phase1"]
announce_experiment("attention_phase1", mode)

if mode_runs(mode):
    source_commit = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"], text=True
    ).strip()
    print(
        f"Phase 1 attention source={source_commit}; resume=True; "
        "full-logit allclose=diagnostic; isolated failures=continue"
    )
    model_result_root = Path(RESULT_DIR) / slugify(MODEL_NAME)
    feature_main_runs = []
    for variant_name in ABLATION_VARIANTS:
        matches = sorted(model_result_root.glob(
            f"l5_feature_ablation_v1_main_{variant_name}/*/raw_results.jsonl"
        ))
        if not matches:
            raise FileNotFoundError(
                f"Missing main evaluation for {variant_name}. Run the Level 5 feature evaluation first."
            )
        feature_main_runs.append(matches[-1])

    selection_command = [
        "python", "scripts/select_feature_attention_cases.py",
        "--annotation_root", ABLATION_ROOT,
        "--main_results", *[str(path) for path in feature_main_runs],
        "--output_path", FEATURE_ATTENTION_CASE_MANIFEST,
        "--base_samples", str(FEATURE_ATTENTION_BASE_SAMPLES),
    ]
    subprocess.run(selection_command, check=True)

    attention_command = [
        "python", "scripts/probe_attention_roi.py",
        "--annotation_path", FEATURE_ATTENTION_CASE_MANIFEST,
        "--output_path", FEATURE_ATTENTION_OUTPUT_PATH,
        "--model_name", MODEL_NAME,
        "--expected_transformers_version", "5.9.0",
        "--seed", str(EVAL_SEED),
        "--deterministic",
        "--attn_implementation", "eager",
        "--max_samples", str(FEATURE_ATTENTION_MAX_SAMPLES),
        "--roi_padding", "8",
        "--roi_assignment", "overlap",
        "--roi_padding_sensitivity", "0,4,8,12",
        "--parity_atol", "0.25",
        "--no-require_standard_logits_match",
        "--minimum_standard_top10_overlap", "0.8",
        "--minimum_standard_logits_cosine_similarity", "0.999",
        "--visualization_layer", "-1",
        "--head_reduction", "mean",
        "--empty_cache_each_sample",
        "--resume",
        "--continue_on_error",
        "--log_every", "8",
        "--no-model_loading_progress",
        "--no-verbose_failures",
        "--no-plots",
    ]
    if MODEL_REVISION:
        attention_command.extend(["--model_revision", MODEL_REVISION])
    process = subprocess.Popen(
        attention_command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(
            f"Feature attention calibration failed with exit code {return_code}."
        )

    subprocess.run([
        "python", "scripts/analyze_attention_roi.py",
        "--input_path", FEATURE_ATTENTION_OUTPUT_PATH,
        "--output_dir", FEATURE_ATTENTION_ANALYSIS_DIR,
    ], check=True)
    selection_summary = Path(FEATURE_ATTENTION_CASE_MANIFEST).with_name(
        f"{Path(FEATURE_ATTENTION_CASE_MANIFEST).stem}_summary.json"
    )
    selection_payload = json.loads(selection_summary.read_text())
    selection_audit = {
        "selection_schema": selection_payload.get("selection_schema"),
        "selected_base_sample_ids": selection_payload.get("selected_base_sample_ids"),
        "evaluation_rows": selection_payload.get("evaluation_rows"),
        "mirrored_video_pairs": selection_payload.get("mirrored_video_pairs"),
        "first_mover_counts": (selection_payload.get("selection_totals") or {}).get("first_mover_counts"),
        "pair_outcome_counts": (selection_payload.get("selection_totals") or {}).get("pair_outcome_counts"),
        "structural_validation": selection_payload.get("structural_validation"),
    }
    print("Selection audit:")
    print(json.dumps(selection_audit, indent=2))
    parity_summary = Path(FEATURE_ATTENTION_OUTPUT_PATH).with_name(
        f"{Path(FEATURE_ATTENTION_OUTPUT_PATH).stem}_summary.json"
    )
    parity_payload = json.loads(parity_summary.read_text())
    parity_rows = parity_payload.get("rows", 0)
    probe_audit = {
        "completed_rows": parity_rows,
        "failed_rows": parity_payload.get("failed_rows", 0),
        "failed_eval_ids": parity_payload.get("failed_eval_ids", []),
        "standard_first_token_matches": f"{parity_payload.get('standard_first_token_matches', 0)}/{parity_rows}",
        "standard_logits_allclose": f"{parity_payload.get('standard_logits_allclose', 0)}/{parity_rows}",
        "archived_prediction_matches": f"{parity_payload.get('archived_prediction_matches', 0)}/{parity_rows}",
        "maximum_logit_absolute_difference": parity_payload.get("maximum_standard_logits_absolute_difference"),
        "minimum_top10_token_overlap": parity_payload.get("minimum_standard_top10_token_overlap"),
        "mean_probe_time_sec": round(parity_payload.get("mean_probe_time_sec", 0), 2),
        "total_wall_time_min": round(parity_payload.get("total_wall_time_sec", 0) / 60, 1),
        "resumed_rows": parity_payload.get("resumed_rows", 0),
        "previous_checkpoint_quarantined": bool(parity_payload.get("quarantined_resume_output")),
        "failure_manifest": parity_payload.get("failure_manifest"),
    }
    print("Probe parity and runtime:")
    print(json.dumps(probe_audit, indent=2))
    for figure_name in [
        "feature_stage_mass_vs_enrichment.png",
        "feature_boundary_mass_vs_enrichment.png",
        "feature_first_mover_mass_vs_enrichment.png",
    ]:
        display(Image(filename=str(Path(FEATURE_ATTENTION_ANALYSIS_DIR) / figure_name)))
elif mode == "analyze":
    require_paths(
        "attention_phase1",
        [FEATURE_ATTENTION_CASE_MANIFEST, FEATURE_ATTENTION_OUTPUT_PATH],
        hint="Restore Phase 1 raw attention artifacts before CPU analysis.",
    )
    subprocess.run([
        "python", "scripts/analyze_attention_roi.py",
        "--input_path", FEATURE_ATTENTION_OUTPUT_PATH,
        "--output_dir", FEATURE_ATTENTION_ANALYSIS_DIR,
    ], check=True)
    print("Rebuilt Phase 1 CPU analysis:", FEATURE_ATTENTION_ANALYSIS_DIR)
elif mode == "reuse":
    require_paths(
        "attention_phase1",
        [
            FEATURE_ATTENTION_CASE_MANIFEST,
            FEATURE_ATTENTION_OUTPUT_PATH,
            Path(FEATURE_ATTENTION_OUTPUT_PATH).with_name(
                f"{Path(FEATURE_ATTENTION_OUTPUT_PATH).stem}_summary.json"
            ),
            Path(FEATURE_ATTENTION_ANALYSIS_DIR) / "summary.json",
            Path(FEATURE_ATTENTION_ANALYSIS_DIR) / "feature_stage_mass_vs_enrichment.png",
        ],
        hint="Restore the completed Phase 1 archive or set attention_phase1=run.",
    )
    print("Reusing Phase 1 attention results without loading Qwen.")
else:
    print("Phase 1 attention calibration skipped.")


## Download Timestamped Experiment Archive

Run this after an experiment. It packages all saved evaluation results, analyses, and ablation annotations into a timestamped ZIP and downloads it to your local computer. Videos are excluded to keep the archive manageable.


In [ ]:
from datetime import datetime
import json
from pathlib import Path
import subprocess
import zipfile
from google.colab import files

project_root = PROJECT_ROOT
source_commit = subprocess.check_output(
    ["git", "-C", str(project_root), "rev-parse", "HEAD"], text=True
).strip()
attention_output = Path(globals().get("ATTENTION_OUTPUT_PATH", ""))
attention_figure_dir = Path(globals().get("ATTENTION_VISUALIZATION_DIR", ""))
attention_analysis_dir = Path(globals().get("ATTENTION_ANALYSIS_DIR", ""))
feature_attention_output = Path(globals().get("FEATURE_ATTENTION_OUTPUT_PATH", ""))
feature_attention_manifest = Path(globals().get("FEATURE_ATTENTION_CASE_MANIFEST", ""))
feature_attention_analysis = Path(globals().get("FEATURE_ATTENTION_ANALYSIS_DIR", ""))
perturbation_root = Path(globals().get("PERTURBATION_ROOT", ""))
perturbation_analysis = Path(globals().get("PERTURBATION_ANALYSIS_DIR", ""))
perturbation_runs = list(
    Path(RESULT_DIR).glob("*/l5_clear_small_many_perturbation/*/raw_results.jsonl")
)
missing_probe_artifacts = []

if EXPERIMENT_MODES["attention_phase0"] != "skip":
    attention_case_manifest = Path(globals().get("ATTENTION_CASE_MANIFEST", ""))
    if not attention_case_manifest.is_file():
        missing_probe_artifacts.append(str(attention_case_manifest))
    if not attention_output.is_file():
        missing_probe_artifacts.append(str(attention_output))
    attention_summary = attention_output.with_name(f"{attention_output.stem}_summary.json")
    if not attention_summary.is_file():
        missing_probe_artifacts.append(str(attention_summary))
    if not attention_figure_dir.is_dir() or not list(attention_figure_dir.glob("*.png")):
        missing_probe_artifacts.append(f"{attention_figure_dir}/*.png")
    for filename in [
        "layer_roi_metrics.csv",
        "layer_target_contrasts.csv",
        "stage_target_contrasts.csv",
        "attention_archive_audit.csv",
        "summary.json",
    ]:
        path = attention_analysis_dir / filename
        if not path.is_file():
            missing_probe_artifacts.append(str(path))
if EXPERIMENT_MODES["attention_phase1"] != "skip":
    feature_required = [
        feature_attention_manifest,
        feature_attention_manifest.with_name(f"{feature_attention_manifest.stem}_summary.json"),
        feature_attention_output,
        feature_attention_output.with_name(f"{feature_attention_output.stem}_summary.json"),
        feature_attention_output.with_name(f"{feature_attention_output.stem}_config.json"),
    ]
    feature_required.extend(
        feature_attention_analysis / filename
        for filename in [
            "layer_roi_metrics.csv",
            "layer_target_contrasts.csv",
            "stage_target_contrasts.csv",
            "paired_stage_target_contrasts.csv",
            "feature_stage_summary.csv",
            "feature_boundary_stage_summary.csv",
            "feature_mover_stage_summary.csv",
            "feature_prompt_stage_summary.csv",
            "feature_pair_outcome_stage_summary.csv",
            "attention_archive_audit.csv",
            "summary.json",
            "feature_stage_mass_vs_enrichment.png",
            "feature_boundary_mass_vs_enrichment.png",
            "feature_first_mover_mass_vs_enrichment.png",
        ]
    )
    for path in feature_required:
        if not path.is_file():
            missing_probe_artifacts.append(str(path))
if EXPERIMENT_MODES["roi_perturbation"] != "skip":
    for filename in ["annotations.jsonl", "manifest.json", "perturbation_stats.jsonl"]:
        path = perturbation_root / filename
        if not path.is_file():
            missing_probe_artifacts.append(str(path))
    for condition in ["visual_boundary", "temporal_boundary"]:
        preview_path = perturbation_analysis / f"roi_qa_{condition}.png"
        if not preview_path.is_file():
            missing_probe_artifacts.append(str(preview_path))
if EXPERIMENT_MODES["roi_perturbation"] != "skip":
    if not perturbation_runs:
        missing_probe_artifacts.append(
            f"{RESULT_DIR}/*/l5_clear_small_many_perturbation/*/raw_results.jsonl"
        )
    perturbation_summary = perturbation_analysis / "summary.json"
    if not perturbation_summary.is_file():
        missing_probe_artifacts.append(str(perturbation_summary))

if missing_probe_artifacts:
    raise FileNotFoundError(
        "Optional probes were enabled but their outputs are missing. "
        "Run the corresponding probe cells successfully before archiving:\n- "
        + "\n- ".join(missing_probe_artifacts)
    )

attention_figures = (
    list(attention_figure_dir.glob("*.png")) if attention_figure_dir.is_dir() else []
)
feature_probe_summary_path = feature_attention_output.with_name(
    f"{feature_attention_output.stem}_summary.json"
)
feature_probe_summary = (
    json.loads(feature_probe_summary_path.read_text())
    if feature_probe_summary_path.is_file()
    else {}
)
probe_status = {
    "roi_mode": EXPERIMENT_MODES["roi_perturbation"],
    "attention_phase0_mode": EXPERIMENT_MODES["attention_phase0"],
    "attention_phase1_mode": EXPERIMENT_MODES["attention_phase1"],
    "feature_attention_expected_rows": globals().get("FEATURE_ATTENTION_MAX_SAMPLES"),
    "feature_attention_completed_rows": feature_probe_summary.get("rows"),
    "feature_attention_resumed_rows": feature_probe_summary.get("resumed_rows"),
    "feature_attention_failed_rows": feature_probe_summary.get("failed_rows"),
    "feature_attention_failed_eval_ids": feature_probe_summary.get("failed_eval_ids", []),
    "feature_attention_analysis_file_count": (
        len(list(feature_attention_analysis.glob("*")))
        if feature_attention_analysis.is_dir() else 0
    ),
    "perturbation_run_count": len(perturbation_runs),
    "attention_figure_count": len(attention_figures),
    "attention_analysis_table_count": (
        len(list(attention_analysis_dir.glob("*.csv")))
        if attention_analysis_dir.is_dir() else 0
    ),
}
archive_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
archive_path = Path("/content") / f"vlm_event_boundary_results_{archive_timestamp}.zip"
manifest = {
    "artifact_type": "real",
    "created_at": archive_timestamp,
    "source_commit": source_commit,
    "model_name": MODEL_NAME,
    "model_revision": MODEL_REVISION or None,
    "eval_seed": EVAL_SEED,
    "deterministic": True,
    "attention_implementation": ATTN_IMPLEMENTATION,
    "ladder_version": DATASET_VERSION,
    "ablation_version": globals().get("ABLATION_VERSION"),
    "pipeline_profile": PIPELINE_PROFILE,
    "experiment_modes": EXPERIMENT_MODES,
    "restored_artifact_archives": [
        {key: value for key, value in item.items() if key != "manifest"}
        for item in RESTORED_ARTIFACTS
    ],
    "optional_probes": probe_status,
}
manifest["config_fingerprint"] = stable_fingerprint({
    "model_name": MODEL_NAME,
    "model_revision": MODEL_REVISION,
    "eval_seed": EVAL_SEED,
    "dataset_version": DATASET_VERSION,
    "experiment_modes": EXPERIMENT_MODES,
})

with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.writestr("archive_manifest.json", json.dumps(manifest, indent=2))
    for folder in [Path(RESULT_DIR), project_root / "analysis"]:
        if folder.exists():
            for path in folder.rglob("*"):
                if path.is_file():
                    archive.write(path, path.relative_to(project_root))
    ablation_root = Path(globals().get("ABLATION_ROOT", ""))
    if ablation_root.exists():
        for pattern in ["annotations.jsonl", "config.json", "README.md"]:
            for path in ablation_root.rglob(pattern):
                archive.write(path, path.relative_to(project_root))
    for extra_data_root in [project_root / "data" / "diagnostics", project_root / "data" / "perturbations"]:
        if extra_data_root.exists():
            for pattern in ["annotations.jsonl", "manifest.json", "perturbation_stats.jsonl"]:
                for path in extra_data_root.rglob(pattern):
                    archive.write(path, path.relative_to(project_root))

print("Optional probe status:", json.dumps(probe_status, indent=2))
print(f"Created {archive_path} ({archive_path.stat().st_size / 1024 / 1024:.1f} MB)")
files.download(str(archive_path))
